# EPL448 – Deliverable 3: Predictive Model Development & Performance Evaluation
## CERN Electron Collision Dataset – Team 2
**Members:** Varnavas Tryfonos, Thrasos Sazeidis, Andreas Evagorou  
**Dataset:** [CERN Electron Collision Data (Kaggle)](https://www.kaggle.com/datasets/fedesoriano/cern-electron-collision-data)

---
## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn – core
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    KFold, GridSearchCV
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, FunctionTransformer
)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

# Models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error, make_scorer
)

# XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed. Install with: pip install xgboost')

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
PALETTE = sns.color_palette('deep')

RANDOM_STATE = 42
print('All imports successful.')

---
## 1. Load Dataset & Reproduce Preprocessing from Deliverable 2

In [ ]:
# Load data
df = pd.read_csv('dielectron.csv')  # <-- adjust path as needed
print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# Drop identifier columns (non-predictive)
df_clean = df.drop(columns=['Run', 'Event'], errors='ignore').copy()
df_clean = df_clean.dropna(subset=['M']).reset_index(drop=True)
print(f'Working dataset: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

### 1.1 Feature Engineering Functions (from Deliverable 2)

In [ ]:
LOG_FEATURES = ['E1', 'E2', 'pt1', 'pt2']  # strictly positive, skewed

def add_engineered_features(df_in):
    """Add physics-motivated engineered features."""
    df_out = df_in.copy()
    df_out['delta_eta']     = df_out['eta1'] - df_out['eta2']
    df_out['delta_phi']     = np.abs(df_out['phi1'] - df_out['phi2'])
    df_out['delta_phi']     = df_out['delta_phi'].apply(lambda x: min(x, 2*np.pi - x))
    df_out['delta_R']       = np.sqrt(df_out['delta_eta']**2 + df_out['delta_phi']**2)
    df_out['sum_pt']        = df_out['pt1'] + df_out['pt2']
    df_out['sum_E']         = df_out['E1'] + df_out['E2']
    df_out['opposite_sign'] = ((df_out['Q1'] * df_out['Q2']) < 0).astype(int)
    return df_out

### 1.2 Revisions from Deliverable 2

**Key revision based on feedback:** In Deliverable 2, all dataset versions were scaled using `StandardScaler`.
The professor noted that tree-based models (Random Forest, XGBoost) are **scale-invariant** and do not benefit from scaling.

**Changes made:**
1. Scaling is no longer pre-applied to dataset versions. Instead, it is included as a **pipeline step** so that:
   - It is only applied when the model requires it (KNN, SVR) or before PCA.
   - It is fitted only on the training folds during cross-validation (no data leakage).
2. All preprocessing (log-transform, scaling, PCA) is now handled **inside scikit-learn Pipelines** fed to `GridSearchCV`.
3. PCA (which uses SVD internally) is now a proper pipeline step in V5, ensuring it is fitted only on training data.
4. Model evaluation in Deliverable 2 was preliminary and is now replaced with the proper workflow: default models → screening → selection → pipeline tuning.

### 1.3 Build Dataset Versions (Unscaled)

We prepare the raw feature matrices for each version. **Scaling and PCA are NOT applied here** — they
are handled inside pipelines where needed.

| Version | Features | Target | PCA | Purpose |
|---------|----------|--------|-----|--------|
| V1 | Original (raw) | M | No | Baseline |
| V2 | Log-transformed energy | M | No | Reduce skewness |
| V3 | Log-transformed energy | log(M) | No | Log target |
| V4 | Engineered + log energy | M | No | Domain knowledge features |
| V5 | Log-transformed energy | M | Yes (in pipeline) | Dimensionality reduction via PCA/SVD |

In [ ]:
# ---- Targets ----
y = df_clean['M'].values
y_log = np.log(y)

# ---- V1: Baseline (raw features, no transforms) ----
base_cols = [c for c in df_clean.columns if c != 'M']
X_v1 = df_clean[base_cols].copy()

# ---- V2: Log-transformed energy features, raw target ----
X_v2 = X_v1.copy()
for f in LOG_FEATURES:
    X_v2[f'log_{f}'] = np.log1p(X_v2[f])
    X_v2 = X_v2.drop(columns=[f])

# ---- V3: Log-transformed energy features + log target ----
# Same features as V2, but target = log(M)
X_v3 = X_v2.copy()

# ---- V4: Engineered features + log-transformed energy ----
X_v4 = add_engineered_features(df_clean.drop(columns=['M']))
for f in LOG_FEATURES:
    X_v4[f'log_{f}'] = np.log1p(X_v4[f])
    X_v4 = X_v4.drop(columns=[f])

# ---- V5: Same features as V2, but PCA applied inside the pipeline ----
# PCA (which uses SVD internally) will be fitted only on training folds
# to avoid data leakage. We store the same raw features as V2 here.
X_v5 = X_v2.copy()

# ---- Stratified train/test split ----
mass_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

datasets_raw = {}  # Unscaled dataset versions
for name, X_df, y_arr in [
    ('V1', X_v1, y), ('V2', X_v2, y),
    ('V3', X_v3, y_log), ('V4', X_v4, y),
    ('V5', X_v5, y)
]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_df.values, y_arr, test_size=0.2,
        random_state=RANDOM_STATE, stratify=mass_bins
    )
    datasets_raw[name] = {
        'X_train': X_tr, 'X_test': X_te,
        'y_train': y_tr, 'y_test': y_te,
        'feature_names': list(X_df.columns),
        'log_target': (name == 'V3'),
        'pca': (name == 'V5')
    }
    print(f'{name}: train={X_tr.shape}, test={X_te.shape}, '
          f'log_target={name=="V3"}, pca={name=="V5"}')

print(f'\nFeature counts: V1={len(X_v1.columns)}, V2={len(X_v2.columns)}, '
      f'V3={len(X_v3.columns)}, V4={len(X_v4.columns)}, V5={len(X_v5.columns)}')

---
## 2. Model Evaluation Metrics

Since this is a **regression** problem (predicting invariant mass M in GeV), we use the following metrics:

| Metric | Formula | Why we use it |
|--------|---------|---------------|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penalises large errors more heavily. Important because errors on Z boson events (~91 GeV) are physically significant. Same units as target (GeV). |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Robust to outliers. Gives a sense of typical prediction error in GeV. |
| **R²** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | Proportion of variance explained. Easy to compare across studies. 1.0 = perfect, 0.0 = baseline (predict mean). |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Relative error — important because M spans a wide range (2–110 GeV). A 5 GeV error means very different things at M=3 GeV vs M=91 GeV. |

**Primary metric for model selection:** R² (used for cross-validation ranking).  
**Secondary metrics:** RMSE and MAPE (reported for interpretation).

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute all regression evaluation metrics."""
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE':  mean_absolute_error(y_true, y_pred),
        'R2':   r2_score(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

# Scorer for cross-validation (R² is the primary metric)
cv_scoring = {
    'R2':   'r2',
    'RMSE': 'neg_root_mean_squared_error',
    'MAE':  'neg_mean_absolute_error'
}

print('Evaluation metrics defined.')

---
## 3. Initial Model Experimentation – Default Hyperparameters

We train all four models with **default hyperparameters** on all five dataset versions using
**5-fold cross-validation**. The goal is to identify which model × dataset combinations
perform best before investing in hyperparameter tuning.

**Pipeline logic:**
- Models that require scaling (KNN, SVR) always get `StandardScaler` in the pipeline.
- Tree-based models (RF, XGBoost) do NOT get scaling (they are scale-invariant).
- V5 (PCA) always gets `StandardScaler → PCA(95% variance)` regardless of model, since PCA requires centred/scaled data.

In [ ]:
def build_pipeline(model_name, use_pca=False):
    """
    Build a pipeline for the given model.
    - KNN/SVR: StandardScaler -> (PCA) -> Model
    - RF/XGB:  Model only (no scaling), unless PCA is needed
    - PCA always requires StandardScaler before it
    """
    steps = []
    
    # Scaling: needed for KNN/SVR, and always before PCA
    if model_name in ('KNN', 'SVR') or use_pca:
        steps.append(('scaler', StandardScaler()))
    
    # PCA (uses SVD internally): retain 95% of variance
    if use_pca:
        steps.append(('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)))
    
    # Model
    if model_name == 'KNN':
        steps.append(('model', KNeighborsRegressor(n_jobs=-1)))
    elif model_name == 'SVR':
        steps.append(('model', SVR()))
    elif model_name == 'RF':
        steps.append(('model', RandomForestRegressor(
            random_state=RANDOM_STATE, n_jobs=-1)))
    elif model_name == 'XGB':
        if XGBOOST_AVAILABLE:
            steps.append(('model', xgb.XGBRegressor(
                random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)))
        else:
            steps.append(('model', GradientBoostingRegressor(
                random_state=RANDOM_STATE)))
    else:
        raise ValueError(f'Unknown model: {model_name}')
    
    return Pipeline(steps)

# Show example pipeline structures
print('=== Pipeline Examples ===')
for name in ['KNN', 'RF', 'XGB']:
    print(f'\n{name} (no PCA):')
    print(f'  {build_pipeline(name)}')
    print(f'{name} (with PCA/SVD):')
    print(f'  {build_pipeline(name, use_pca=True)}')

In [ ]:
# Run 5-fold cross-validation for every model × dataset version combination
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# SVR is O(n^2-n^3), so we subsample for the screening phase
N_SVR_SCREEN = 20000

all_cv_results = []

for ver_name, ver_data in datasets_raw.items():
    X_tr = ver_data['X_train']
    y_tr = ver_data['y_train']
    use_pca = ver_data['pca']
    
    for model_name in ['KNN', 'SVR', 'RF', 'XGB']:
        # Build pipeline with PCA flag from dataset version
        pipe = build_pipeline(model_name, use_pca=use_pca)
        
        # SVR subsample for speed
        if model_name == 'SVR':
            n = min(N_SVR_SCREEN, len(X_tr))
            idx = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), n, replace=False)
            X_cv, y_cv = X_tr[idx], y_tr[idx]
        else:
            X_cv, y_cv = X_tr, y_tr
        
        # Cross-validate
        cv_results = cross_validate(
            pipe, X_cv, y_cv, cv=kf,
            scoring=cv_scoring, n_jobs=-1,
            return_train_score=False
        )
        
        result = {
            'Model': model_name,
            'Dataset': ver_name,
            'R2_mean':   cv_results['test_R2'].mean(),
            'R2_std':    cv_results['test_R2'].std(),
            'RMSE_mean': -cv_results['test_RMSE'].mean(),  # negate back
            'RMSE_std':  cv_results['test_RMSE'].std(),
            'MAE_mean':  -cv_results['test_MAE'].mean(),
            'MAE_std':   cv_results['test_MAE'].std()
        }
        all_cv_results.append(result)
        pca_tag = ' [PCA]' if use_pca else ''
        print(f'{model_name:4s} x {ver_name}{pca_tag}: '
              f'R²={result["R2_mean"]:.4f} (±{result["R2_std"]:.4f})  '
              f'RMSE={result["RMSE_mean"]:.3f}  MAE={result["MAE_mean"]:.3f}')
    
    print()  # blank line between dataset versions

cv_df = pd.DataFrame(all_cv_results)
print('Screening complete.')

### 3.1 Screening Results Table

In [ ]:
# Pivot table: rows = models, columns = dataset versions, values = R²
pivot_r2 = cv_df.pivot(index='Model', columns='Dataset', values='R2_mean')
pivot_r2 = pivot_r2[['V1', 'V2', 'V3', 'V4', 'V5']]  # order columns
pivot_r2 = pivot_r2.loc[['KNN', 'SVR', 'RF', 'XGB']]  # order rows

print('=== Cross-Validated R² Scores (Default Hyperparameters) ===')
print()
display(pivot_r2.style.format('{:.4f}').highlight_max(axis=None, color='lightgreen'))

# Also show RMSE
pivot_rmse = cv_df.pivot(index='Model', columns='Dataset', values='RMSE_mean')
pivot_rmse = pivot_rmse[['V1', 'V2', 'V3', 'V4', 'V5']]
pivot_rmse = pivot_rmse.loc[['KNN', 'SVR', 'RF', 'XGB']]

print('\n=== Cross-Validated RMSE (Default Hyperparameters) ===')
print()
display(pivot_rmse.style.format('{:.3f}').highlight_min(axis=None, color='lightgreen'))

In [ ]:
# Visual comparison of screening results
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

model_colors = {'KNN': 'steelblue', 'SVR': 'darkorange', 'RF': 'seagreen', 'XGB': 'crimson'}

# R² bar plot
ax = axes[0]
bar_data = cv_df.copy()
bar_data['label'] = bar_data['Model'] + ' / ' + bar_data['Dataset']
bar_data = bar_data.sort_values('R2_mean', ascending=True)
colors = [model_colors[m] for m in bar_data['Model']]
ax.barh(bar_data['label'], bar_data['R2_mean'], xerr=bar_data['R2_std'],
        color=colors, edgecolor='none', capsize=3)
ax.set_xlabel('R² (5-fold CV)', fontsize=11)
ax.set_title('Model × Dataset Screening – R²', fontsize=13)
ax.axvline(0, color='grey', linewidth=0.5)

# RMSE bar plot
ax = axes[1]
bar_data = cv_df.sort_values('RMSE_mean', ascending=False)
colors = [model_colors[m] for m in bar_data['Model']]
ax.barh(bar_data['Model'] + ' / ' + bar_data['Dataset'], bar_data['RMSE_mean'],
        xerr=bar_data['RMSE_std'], color=colors, edgecolor='none', capsize=3)
ax.set_xlabel('RMSE in GeV (5-fold CV)', fontsize=11)
ax.set_title('Model × Dataset Screening – RMSE', fontsize=13)

# Add legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=m) for m, c in model_colors.items()]
axes[0].legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('fig_screening_results.png', bbox_inches='tight')
plt.show()

---
## 4. Selection of Best-Performing Models & Dataset Versions

Based on the screening results above, we select the **top 2–3 models** and **top 2 dataset versions**
for hyperparameter tuning.

In [ ]:
# Identify top model-dataset combinations by R²
top_combos = cv_df.nlargest(8, 'R2_mean')[['Model', 'Dataset', 'R2_mean', 'RMSE_mean']]
print('Top 8 model × dataset combinations by R²:')
display(top_combos)

# Identify best models (by average R² across all datasets)
model_avg = cv_df.groupby('Model')['R2_mean'].mean().sort_values(ascending=False)
print('\nAverage R² by model (across all datasets):')
print(model_avg.to_string())

# Identify best datasets (by average R² across all models)
dataset_avg = cv_df.groupby('Dataset')['R2_mean'].mean().sort_values(ascending=False)
print('\nAverage R² by dataset (across all models):')
print(dataset_avg.to_string())

# ============================================================
# SET THESE BASED ON YOUR ACTUAL RESULTS:
# ============================================================
SELECTED_MODELS   = ['XGB', 'RF']       # <-- UPDATE after reviewing screening
SELECTED_DATASETS = ['V4', 'V2']        # <-- UPDATE after reviewing screening
# ============================================================

print(f'\n>>> Selected models for tuning:   {SELECTED_MODELS}')
print(f'>>> Selected datasets for tuning: {SELECTED_DATASETS}')

---
## 5. Pipeline Definition

We construct **scikit-learn Pipelines** that integrate preprocessing with the model.
This ensures that:
- Preprocessing (scaling, PCA) is fitted **only on training folds** during cross-validation.
- `GridSearchCV` handles train/validation splitting internally.
- There is **no data leakage** from test data into preprocessing.

**Pipeline structures by model and dataset version:**

| Model | V1–V4 (no PCA) | V5 (with PCA/SVD) |
|-------|----------------|-------------------|
| KNN   | StandardScaler → KNN | StandardScaler → PCA(95%) → KNN |
| SVR   | StandardScaler → SVR | StandardScaler → PCA(95%) → SVR |
| RF    | RF (no preprocessing) | StandardScaler → PCA(95%) → RF |
| XGB   | XGB (no preprocessing) | StandardScaler → PCA(95%) → XGB |

In [ ]:
# Show the actual pipelines that will be used for tuning
print('=== Pipelines for Selected Combinations ===')
for model_name in SELECTED_MODELS:
    for dataset_name in SELECTED_DATASETS:
        use_pca = datasets_raw[dataset_name]['pca']
        pipe = build_pipeline(model_name, use_pca=use_pca)
        print(f'\n{model_name} on {dataset_name} (PCA={use_pca}):')
        for step_name, step_obj in pipe.steps:
            print(f'  → {step_name}: {type(step_obj).__name__}')

---
## 6. Hyperparameter Tuning with GridSearchCV

For each selected model × dataset combination, we define a hyperparameter grid and
run `GridSearchCV` with 5-fold cross-validation. The pipeline ensures preprocessing
is applied correctly within each fold.

In [ ]:
# Define hyperparameter grids
# Note: pipeline parameter names use the format 'step_name__param_name'

param_grids = {
    'KNN': {
        'model__n_neighbors': [3, 5, 10, 15, 20],
        'model__weights': ['uniform', 'distance'],
        'model__metric': ['euclidean', 'manhattan']
    },
    'SVR': {
        'model__C': [1, 10, 100],
        'model__epsilon': [0.1, 0.5, 1.0],
        'model__kernel': ['rbf'],
        'model__gamma': ['scale', 'auto']
    },
    'RF': {
        'model__n_estimators': [100, 300, 500],
        'model__max_depth': [10, 20, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    },
    'XGB': {
        'model__n_estimators': [100, 300, 500],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__max_depth': [3, 6, 10],
        'model__subsample': [0.8, 1.0],
        'model__colsample_bytree': [0.8, 1.0]
    }
}

# Print grid sizes
for name in SELECTED_MODELS:
    grid = param_grids[name]
    n_combos = 1
    for v in grid.values():
        n_combos *= len(v)
    print(f'{name}: {n_combos} combinations × 5 folds = {n_combos * 5} fits')

In [ ]:
# Run GridSearchCV for each selected model × dataset
grid_results = {}

for model_name in SELECTED_MODELS:
    for dataset_name in SELECTED_DATASETS:
        key = f'{model_name}_{dataset_name}'
        print(f'\n{"="*60}')
        print(f'Tuning: {model_name} on {dataset_name}')
        print(f'{"="*60}')
        
        ver = datasets_raw[dataset_name]
        X_tr = ver['X_train']
        y_tr = ver['y_train']
        use_pca = ver['pca']
        
        # Build pipeline (with PCA if V5)
        pipe = build_pipeline(model_name, use_pca=use_pca)
        
        # For SVR, subsample to keep runtime manageable
        if model_name == 'SVR':
            n = min(N_SVR_SCREEN, len(X_tr))
            idx = np.random.RandomState(RANDOM_STATE).choice(len(X_tr), n, replace=False)
            X_search, y_search = X_tr[idx], y_tr[idx]
        else:
            X_search, y_search = X_tr, y_tr
        
        # GridSearchCV
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grids[model_name],
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='r2',
            n_jobs=-1,
            verbose=1,
            return_train_score=True
        )
        gs.fit(X_search, y_search)
        
        grid_results[key] = {
            'grid_search': gs,
            'best_params': gs.best_params_,
            'best_cv_r2':  gs.best_score_,
            'model_name':  model_name,
            'dataset_name': dataset_name
        }
        
        print(f'\nBest params: {gs.best_params_}')
        print(f'Best CV R²:  {gs.best_score_:.4f}')

In [ ]:
# Summary of tuning results
tuning_summary = []
for key, res in grid_results.items():
    tuning_summary.append({
        'Model': res['model_name'],
        'Dataset': res['dataset_name'],
        'Best CV R²': res['best_cv_r2'],
        'Best Params': str(res['best_params'])
    })

tuning_df = pd.DataFrame(tuning_summary).sort_values('Best CV R²', ascending=False)
print('=== Hyperparameter Tuning Summary ===')
display(tuning_df)

---
## 7. Final Model Evaluation on Test Set

We evaluate the best tuned models on the **held-out test set** (which was never seen during
cross-validation or hyperparameter tuning).

In [ ]:
# Evaluate each tuned model on its corresponding test set
final_results = []
predictions = {}  # store for plotting

for key, res in grid_results.items():
    model_name = res['model_name']
    dataset_name = res['dataset_name']
    ver = datasets_raw[dataset_name]
    
    best_model = res['grid_search'].best_estimator_
    
    # Predict on test set
    y_pred = best_model.predict(ver['X_test'])
    y_true = ver['y_test']
    
    # If log-target (V3), back-transform for evaluation
    if ver['log_target']:
        y_pred_eval = np.exp(y_pred)
        y_true_eval = np.exp(y_true)
    else:
        y_pred_eval = y_pred
        y_true_eval = y_true
    
    metrics = compute_metrics(y_true_eval, y_pred_eval)
    metrics['Model'] = model_name
    metrics['Dataset'] = dataset_name
    metrics['Best Params'] = str(res['best_params'])
    final_results.append(metrics)
    
    predictions[key] = {
        'y_true': y_true_eval,
        'y_pred': y_pred_eval,
        'model_name': model_name,
        'dataset_name': dataset_name
    }
    
    print(f'{model_name} / {dataset_name}: R²={metrics["R2"]:.4f}  '
          f'RMSE={metrics["RMSE"]:.3f} GeV  MAE={metrics["MAE"]:.3f} GeV  '
          f'MAPE={metrics["MAPE"]:.2f}%')

final_df = pd.DataFrame(final_results)
print('\n=== Final Test Set Evaluation ===')
display(final_df[['Model', 'Dataset', 'R2', 'RMSE', 'MAE', 'MAPE', 'Best Params']])

### 7.1 Predicted vs Actual Scatter Plots

In [ ]:
n_plots = len(predictions)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

for ax, (key, pred) in zip(axes, predictions.items()):
    # Subsample for readability
    n = len(pred['y_true'])
    idx = np.random.RandomState(42).choice(n, size=min(3000, n), replace=False)
    
    ax.scatter(pred['y_true'][idx], pred['y_pred'][idx],
              alpha=0.2, s=5, color='steelblue', rasterized=True)
    lims = [0, max(pred['y_true'].max(), pred['y_pred'].max()) * 1.05]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual M (GeV)', fontsize=11)
    ax.set_ylabel('Predicted M (GeV)', fontsize=11)
    
    # Get metrics for title
    m = compute_metrics(pred['y_true'], pred['y_pred'])
    ax.set_title(f"{pred['model_name']} / {pred['dataset_name']}\n"
                 f"R²={m['R2']:.4f}  RMSE={m['RMSE']:.2f} GeV", fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.suptitle('Predicted vs Actual – Tuned Models (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig_pred_vs_actual.png', bbox_inches='tight')
plt.show()

### 7.2 Residual Analysis

In [ ]:
n_plots = len(predictions)
fig, axes = plt.subplots(2, n_plots, figsize=(6 * n_plots, 10))
if n_plots == 1:
    axes = axes.reshape(-1, 1)

for j, (key, pred) in enumerate(predictions.items()):
    residuals = pred['y_true'] - pred['y_pred']
    
    # Top row: Residuals vs Actual
    ax = axes[0, j]
    idx = np.random.RandomState(42).choice(len(residuals), min(3000, len(residuals)), replace=False)
    ax.scatter(pred['y_true'][idx], residuals[idx], alpha=0.2, s=5, color='steelblue', rasterized=True)
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Actual M (GeV)', fontsize=10)
    ax.set_ylabel('Residual (GeV)', fontsize=10)
    ax.set_title(f"{pred['model_name']} / {pred['dataset_name']}\nResiduals vs Actual", fontsize=11)
    
    # Bottom row: Residual distribution
    ax = axes[1, j]
    ax.hist(residuals, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Residual (GeV)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'Residual Distribution\nMean={residuals.mean():.3f}, Std={residuals.std():.3f}', fontsize=11)

plt.suptitle('Residual Analysis – Tuned Models', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_residual_analysis.png', bbox_inches='tight')
plt.show()

### 7.3 Performance by Mass Region

Since the target M is multimodal (J/ψ ~3.1 GeV, Υ ~9.5 GeV, Z ~91 GeV),
it is important to evaluate how well each model performs in each resonance region.

In [ ]:
# Define mass regions based on physics
def get_mass_region(m):
    if m < 10:
        return 'Low (< 10 GeV)'
    elif m < 50:
        return 'Mid (10–50 GeV)'
    else:
        return 'High (> 50 GeV)'

region_results = []

for key, pred in predictions.items():
    regions = np.array([get_mass_region(m) for m in pred['y_true']])
    
    for region in ['Low (< 10 GeV)', 'Mid (10–50 GeV)', 'High (> 50 GeV)']:
        mask = regions == region
        if mask.sum() == 0:
            continue
        m = compute_metrics(pred['y_true'][mask], pred['y_pred'][mask])
        m['Model'] = pred['model_name']
        m['Dataset'] = pred['dataset_name']
        m['Region'] = region
        m['N_events'] = int(mask.sum())
        region_results.append(m)

region_df = pd.DataFrame(region_results)
print('=== Performance by Mass Region ===')
display(region_df[['Model', 'Dataset', 'Region', 'N_events', 'R2', 'RMSE', 'MAE', 'MAPE']]
        .style.format({'R2': '{:.4f}', 'RMSE': '{:.3f}', 'MAE': '{:.3f}', 'MAPE': '{:.2f}'}))

### 7.4 Feature Importance (Best Model)

In [ ]:
# Extract feature importance from the overall best model
best_key = final_df.loc[final_df['R2'].idxmax()]
best_combo_key = f"{best_key['Model']}_{best_key['Dataset']}"
best_gs = grid_results[best_combo_key]['grid_search']
best_estimator = best_gs.best_estimator_

# Get the actual model from the pipeline
if hasattr(best_estimator, 'named_steps'):
    actual_model = best_estimator.named_steps['model']
else:
    actual_model = best_estimator

# Check if PCA was used – if so, feature importances map to PCA components, not original features
best_dataset = best_key['Dataset']
uses_pca = datasets_raw[best_dataset]['pca']

if hasattr(actual_model, 'feature_importances_'):
    importances = actual_model.feature_importances_
    
    if uses_pca:
        # Feature importances are for PCA components
        feat_names = [f'PC{i+1}' for i in range(len(importances))]
        title_suffix = '(PCA Components)'
    else:
        # Feature importances are for original features
        feat_names = datasets_raw[best_dataset]['feature_names']
        title_suffix = '(Original Features)'
    
    feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.head(15).plot(kind='barh', ax=ax, color='steelblue', edgecolor='none')
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance (MDI)', fontsize=12)
    ax.set_title(f'Feature Importances – {best_key["Model"]} / {best_dataset} {title_suffix}', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig_feature_importance_best.png', bbox_inches='tight')
    plt.show()
    
    print('Top 10 features:')
    print(feat_imp.head(10).to_string())
else:
    print(f'Feature importances not available for {type(actual_model).__name__}')

---
## 8. Comparison of Default vs Tuned Performance

In [ ]:
# Compare default CV R² with tuned CV R² for selected combinations
comparison_rows = []

for key, res in grid_results.items():
    model_name = res['model_name']
    dataset_name = res['dataset_name']
    
    # Find default R² from screening
    default_r2 = cv_df[
        (cv_df['Model'] == model_name) & (cv_df['Dataset'] == dataset_name)
    ]['R2_mean'].values[0]
    
    tuned_r2 = res['best_cv_r2']
    
    comparison_rows.append({
        'Model': model_name,
        'Dataset': dataset_name,
        'Default R² (CV)': default_r2,
        'Tuned R² (CV)': tuned_r2,
        'Improvement': tuned_r2 - default_r2
    })

comp_df = pd.DataFrame(comparison_rows)
print('=== Default vs Tuned Performance ===')
display(comp_df.style.format({
    'Default R² (CV)': '{:.4f}',
    'Tuned R² (CV)': '{:.4f}',
    'Improvement': '{:+.4f}'
}))

---
## 9. Discussion

### 9.1 Key Findings

*(Fill in after running the experiments. Template below.)*

- **Best performing model:** [Model] on dataset [V?] achieved R² = [?], RMSE = [?] GeV.
- **Effect of preprocessing:** Log-transformation of energy features [improved/did not improve] performance for [models]. Engineered physics features (V4) [helped/did not help] because...
- **Effect of PCA (V5):** Dimensionality reduction via PCA (SVD) [improved/reduced] performance for [models]. This is because [PCA decorrelated features and helped distance-based models / PCA lost information that tree-based models could use directly].
- **Scaling insight:** As noted in the Deliverable 2 feedback, removing unnecessary scaling for tree-based models [had no effect / slightly improved training speed].
- **Mass region performance:** Models performed best in the [high/low/mid] mass region because...

### 9.2 Comparison with Literature

| Aspect | Kilic et al. (2023) | Our Study |
|--------|--------------------|-----------|
| Dataset | CERN dielectron (same) | CERN dielectron (same) |
| Target | Z boson mass | Invariant mass M (full range) |
| Best Algorithm | [?] | [?] |
| Features Used | [?] | Original + engineered physics features |
| Dimensionality Reduction | [?] | PCA (95% variance, V5) |
| Best R² | [?] | [?] |
| Best RMSE | [?] | [?] GeV |

### 9.3 Domain-Specific Validation

*(Discuss: Do the model's predictions make physical sense? Does it correctly identify resonance peaks? Are the most important features physically meaningful? How does PCA relate to the underlying physics — do the principal components correspond to meaningful kinematic quantities?)*

### 9.4 Study Limitations

- SVR training required subsampling to 20K events due to O(n²) complexity
- The multimodal target distribution (3 resonance peaks) makes it challenging for models optimising MSE
- PCA reduces interpretability since features become linear combinations of original kinematic variables
- [Add more based on your findings]

---
## 10. Final Summary

In [ ]:
print('=' * 75)
print('DELIVERABLE 3 – FINAL SUMMARY')
print('=' * 75)
print()
print('Dataset: CERN Electron Collision – 100,000 dielectron events')
print('Task:    Regression – predict invariant mass M (GeV)')
print()
print(f'Models evaluated:      KNN, SVR, Random Forest, XGBoost')
print(f'Dataset versions:      V1 (baseline), V2 (log features), V3 (log target), '
      f'V4 (engineered), V5 (PCA/SVD)')
print(f'Models tuned:          {SELECTED_MODELS}')
print(f'Datasets tuned on:     {SELECTED_DATASETS}')
print()
print('--- Best Model (Test Set) ---')
best_row = final_df.loc[final_df['R2'].idxmax()]
print(f'Model:     {best_row["Model"]}')
print(f'Dataset:   {best_row["Dataset"]}')
print(f'R²:        {best_row["R2"]:.4f}')
print(f'RMSE:      {best_row["RMSE"]:.3f} GeV')
print(f'MAE:       {best_row["MAE"]:.3f} GeV')
print(f'MAPE:      {best_row["MAPE"]:.2f}%')
print(f'Params:    {best_row["Best Params"]}')
print()
print('=' * 75)

---
## References

[1] McCauley, T. (2014). Events with two electrons from 2010. CERN Open Data Portal. DOI: 10.7483/OPENDATA.CMS.PCSW.AHVG  
[2] Kilic, H., Ozturk, S., & Yildirim, E. (2023). Machine learning model performances for the Z boson mass identification through dielectron decay channel. *The European Physical Journal Plus*, 138(1), 87.  
[3] Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825-2830.  
[4] Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*.  
[5] Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5-32.  
[6] Vapnik, V. (1995). *The Nature of Statistical Learning Theory*. Springer.  
[7] Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Trans. Information Theory*, 13(1), 21-27.  
[8] Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer. DOI: 10.1007/b98835